In [1]:
%pip install -r ../requirements.txt

  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached fastjsonschema-2.21.2-py3-none-any.whl.metadata (2.3 kB)
Using cached nbformat-5.10.4-py3-none-any.whl (78 kB)
Using cached fastjsonschema-2.21.2-py3-none-any.whl (24 kB)

  Attempting uninstall: nbformat

    Found existing installation: nbformat 4.2.0

    Uninstalling nbformat-4.2.0:

      Successfully uninstalled nbformat-4.2.0

   -------------------- ------------------- 1/2 [nbformat]
   ---------------------------------------- 2/2 [nbformat]

Note: you may need to restart the kernel to use updated packages.


## Infinite Solutions

In [2]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

# Uncomment this if you want Plotly to open in the browser
# pio.renderers.default = "browser"


# ------------------------------------------------------------
# Problem setup
# System:
#   x + y + z = 50
#   y + 2z = 60
#
# Parametric solution:
#   x = z - 10
#   y = 60 - 2z
#   z = t
# ------------------------------------------------------------

def parametric_line(t):
    """
    Returns the parametric solution of the intersection line.

    Parameters
    ----------
    t : array-like
        Parameter values.

    Returns
    -------
    x, y, z : arrays
        Coordinates of the line.
    """
    t = np.asarray(t)
    x = t - 10
    y = 60 - 2 * t
    z = t
    return x, y, z


def build_plane_patches(t_vals, s_vals):
    """
    Builds two plane patches around the intersection line.

    Plane 1: x + y + z = 50
    Plane 2: y + 2z = 60

    Parameters
    ----------
    t_vals : array
        Values used along the intersection line direction.
    s_vals : array
        Values used to sweep out each plane around the line.

    Returns
    -------
    (X1, Y1, Z1), (X2, Y2, Z2) : tuples of arrays
        Meshes for Plane 1 and Plane 2.
    """
    T, S = np.meshgrid(t_vals, s_vals)

    # Plane 1
    X1 = (T - 10) + S
    Y1 = (60 - 2 * T) - S
    Z1 = T

    # Plane 2
    X2 = (T - 10) + S
    Y2 = 60 - 2 * T
    Z2 = T

    return (X1, Y1, Z1), (X2, Y2, Z2)


def add_planes(fig, plane1, plane2, opacity=0.55):
    """
    Adds the two planes to a Plotly figure.
    """
    X1, Y1, Z1 = plane1
    X2, Y2, Z2 = plane2

    fig.add_surface(
        x=X1,
        y=Y1,
        z=Z1,
        opacity=opacity,
        showscale=False,
        name="Plane 1: x + y + z = 50"
    )

    fig.add_surface(
        x=X2,
        y=Y2,
        z=Z2,
        opacity=opacity,
        showscale=False,
        name="Plane 2: y + 2z = 60"
    )


def style_figure(fig, title, x_range, y_range, z_range):
    """
    Applies common layout settings.
    """
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title='x',
            yaxis_title='y',
            zaxis_title='z',
            xaxis=dict(range=x_range),
            yaxis=dict(range=y_range),
            zaxis=dict(range=z_range),
            aspectmode="data",
            camera=dict(eye=dict(x=1.7, y=1.4, z=1.2))
        ),
        margin=dict(l=0, r=0, t=50, b=0)
    )


def plot_infinite_solutions():
    """
    Plots the full intersection line over the reals.
    """
    # Wider real-valued ranges to show the planes clearly
    t_vals = np.linspace(-20, 20, 30)
    s_vals = np.linspace(-20, 20, 50)

    plane1, plane2 = build_plane_patches(t_vals, s_vals)

    # Line over the same visible t-range
    t_line = np.linspace(-20, 20, 300)
    x_line, y_line, z_line = parametric_line(t_line)

    fig = go.Figure()
    add_planes(fig, plane1, plane2, opacity=0.55)

    fig.add_scatter3d(
        x=x_line,
        y=y_line,
        z=z_line,
        mode='lines',
        name='Intersection line',
        line=dict(color='red', width=10)
    )

    # A sample point on the line
    k = len(t_line) // 2
    fig.add_scatter3d(
        x=[x_line[k]],
        y=[y_line[k]],
        z=[z_line[k]],
        mode='markers',
        name='Point on the line',
        marker=dict(size=6, color='black')
    )

    style_figure(
        fig,
        title="1. Infinite Solutions with x, y, z ∈ ℝ",
        x_range=[-50, 30],
        y_range=[0, 100],
        z_range=[-20, 20]
    )
    pio.write_image(fig, "../figures/infinite_solutions.png")
    fig.show()


def integer_solutions(z_min=10, z_max=30):
    """
    Computes all integer solutions under the constraints:
    z in {z_min, ..., z_max}, x,y,z >= 0.
    """
    z_vals = np.arange(z_min, z_max + 1)
    x_vals = z_vals - 10
    y_vals = 60 - 2 * z_vals

    mask = (x_vals >= 0) & (y_vals >= 0) & (z_vals >= 0)

    return x_vals[mask], y_vals[mask], z_vals[mask]


def plot_finite_solutions():
    """
    Plots the finite set of integer solutions.
    """
    # Same planes, but now we focus on the relevant integer region
    t_vals = np.linspace(10, 30, 30)
    s_vals = np.linspace(-20, 20, 50)

    plane1, plane2 = build_plane_patches(t_vals, s_vals)

    x_int, y_int, z_int = integer_solutions(10, 30)
    n_solutions = len(z_int)

    fig = go.Figure()
    add_planes(fig, plane1, plane2, opacity=0.45)

    # Draw the finite segment as a thin guide
    fig.add_scatter3d(
        x=x_int,
        y=y_int,
        z=z_int,
        mode='lines+markers',
        name='Integer solutions',
        line=dict(color='red', width=6),
        marker=dict(size=5, color='black')
    )

    style_figure(
        fig,
        title=f"2. Finite Solutions ({n_solutions}) with x, y, z ∈ ℤ, x,y,z ≥ 0",
        x_range=[-5, 25],
        y_range=[0, 45],
        z_range=[10, 30]
    )


    pio.write_image(fig, "../figures/finite_solutions.png")
    fig.show()

# Run both plots in sequence
plot_infinite_solutions()
plot_finite_solutions()